# Carbon Mapper products — reference walkthrough

How to get every Carbon Mapper product with `georeader`, in four short
sections: the **product registry**, **per-plume products** (L3A),
**scene tiles** (L2B), and **sources**. Ends with a cheat-sheet of what
is and isn't reachable on the current API.

Two facts shape everything (verified against the live API, 2026-07):

1. **STAC is history-only** — nothing after `v3a` (2025-12). All current
   data is served by the asset proxy, at URLs derived from each plume's
   own record. The reader does this for you; no version is hardcoded.
2. **Plumes are the only door to current data** — there is no way to
   browse recent tiles directly (`/catalog/scenes` is closed, STAC is
   stale). You discover a plume, then reach its products and its parent
   tile from there.


## Levels at a glance

| Level | Entity | What you get | Keyed by | Latest (2026+) | History (≤ 2025-12) |
|---|---|---|---|---|---|
| Plume record | plume | metadata + URL fields (`plume_tif` → L3A-vis, `con_tif` → L3A-ime) | `plume_id` | ✅ `/catalog/plumes/annotated` — **the only discovery route** | ✅ same |
| L3A vis + ime | image | per-plume crops: mask, concentrations, outlines, rgb, IME set | `plume_id` | ✅ asset proxy, URLs derived from the record | ✅ also in STAC (≤ v3a) |
| L2B + l2b-rgb | tile | full-swath cmf, uncertainty, artifact-mask, uas, rgb | scene name (from `plume_id`) | ✅ asset proxy — reachable **only via a plume**; no browsing (`/catalog/scenes` 401, STAC stale) | ✅ STAC (≤ v3a) |
| Sources (L4) | source | DBSCAN site clusters, stats, embedded plume records | `source_name` (re-clustered — never hardcode) | ✅ REST `/catalog/source*`, live | `l4a-*` STAC ≤ v3a, no per-item assets |
| L2C / L3C | detections / attribution | — | — | not exposed by the reader | — |

Publication timing: plume + L3A appear together (same-day to ~30-day
embargo observed); the L2B parent can lag by days-to-weeks and may sit
one version behind a re-versioned L3A.


## Setup

Auth resolves from `CARBONMAPPER_TOKEN`, `CARBONMAPPER_EMAIL`+`PASSWORD`,
or `~/.georeader/auth_carbonmapper.json`. The retry adapter and GDAL
settings keep the notebook polite under Carbon Mapper's rate limits.


In [ ]:
import os

from georeader.readers.carbonmapper import (
    CarbonMapperConfig,
    CMCollectionSpec,
    CMPlumeImage,
    CMProductFamily,
    CMProductNotSelected,
    api_queries,
)
from georeader.readers.carbonmapper import products as P

# 429-resilient HTTP: Carbon Mapper rate-limits per account. Route every
# call in this kernel through a retry-aware session (honours Retry-After).
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

_session = requests.Session()
_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=8, backoff_factor=2.0,
    status_forcelist=(429, 500, 502, 503, 504),
    respect_retry_after_header=True,
    allowed_methods=frozenset(["GET", "POST"]),
)))
requests.get, requests.post, requests.request = (
    _session.get, _session.post, _session.request,
)

TOKEN = CarbonMapperConfig.load().refresh_access_token()

# Rasters stream over GDAL/libcurl — pass the token + retries there too.
os.environ["GDAL_HTTP_HEADERS"] = f"Authorization: Bearer {TOKEN}"
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".tif,.TIF"
os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "3"

# Protagonist: a strong (1954 kg/h) Tanager plume. Its L3A was
# re-versioned to v3d while its L2B parent still serves at v3c — the
# version split the reader resolves automatically.
PLUME_ID = "tan20260331t181625c77s4001-D"
print(f"plume_id = {PLUME_ID}")


## 1 · The product registry

Every product is a descriptor in
`georeader.readers.carbonmapper.products`: its asset key, its
**family** (which collection serves it), and how it opens (raster →
`RasterioReader`, vector → shapely, text → `str`, quicklook → `bytes`).

Families and versioning: a `CMCollectionSpec` — the plume's
`(gas, cmf_type, version)` triple, parsed from its own record —
composes all four collection ids consistently.


In [ ]:
# The registry is the single source of truth — print it rather than
# maintaining a parallel table by hand.
print(f"{'product':30s} {'family':8s} {'opens as':16s} description")
print("-" * 100)
for prod in P.ALL_PRODUCTS:
    kind = type(prod).__name__.removeprefix("CM").removesuffix("Product")
    print(f"{prod.key:30s} {prod.family.value:8s} {kind:16s} {prod.description}")

# One spec pins every collection id consistently (versions come from the
# plume record at run time — nothing here is hardcoded in the reader).
spec = CMCollectionSpec(version="v3d")
print()
for fam in CMProductFamily:
    print(f"{fam.name:8s} -> {spec.collection_id(fam)}")


## 2 · Per-plume products (L3A)

`CMPlumeImage` exposes the products **you select** as lazy properties
(`mask`, `concentrations`, `ime_concentrations`, `ime_mask`, `rgb`,
`outline`, `ime_outline`). Defaults = the 7 GeoTIFF/GeoJSON products;
PNG quicklooks are opt-in via `products=` and `img.product(...)`.

These are **thumbnail-grade** crops (11–100 px per side) — right for
previews and polygons, wrong for analysis. Analysis-grade comes from
the L2B parent in §3.


In [ ]:
# Fetch the typed plume, then build the image bundle from it (no second
# catalog call). The default selection is the 7 GeoTIFF/GeoJSON products.
plume = api_queries.get_plume(TOKEN, PLUME_ID)
em = f"{plume.emission_auto:.0f} kg/h" if plume.emission_auto is not None else "no emission yet"
print(f"{plume.plume_id}  {em}  sector={plume.sector or '—'}")
print(f"collection_spec: {plume.collection_spec}")
print()

img = CMPlumeImage.from_cmrawplume(plume, token=TOKEN)
print(img)

# Selective bundle: ask for exactly what you need. Anything outside the
# selection raises CMProductNotSelected — no silent Nones.
lean = CMPlumeImage.from_cmrawplume(
    plume, token=TOKEN, products=(P.PLUME_OUTLINE, P.RGB_TIF),
)
print(f"\nlean bundle URLs: {sorted(lean.urls)}")
try:
    lean.concentrations
except CMProductNotSelected as exc:
    print(f"lean.concentrations -> CMProductNotSelected: {exc}")


In [ ]:
def show(name, geom):
    if geom is None:
        print(f"{name:14s} (absent)")
    else:
        print(f"{name:14s} area={geom.area:.3e} deg^2  "
              f"bounds={tuple(round(b, 4) for b in geom.bounds)}")

show("outline", img.outline)          # broader plume polygon (canonical GeoJSON)
show("ime_outline", img.ime_outline)  # tighter — the emission_auto integration area


### 2.1 Visualise

The three rasters worth a look, plume outline overlaid. Note the pixel
counts in the titles — this is why they're preview-grade.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyproj import Transformer
from shapely.ops import transform as shp_transform


def plot_outline(ax, geom_4326, dst_crs):
    """Reproject the plume outline into dst_crs and stroke it in red."""
    if geom_4326 is None:
        return
    tr = Transformer.from_crs("EPSG:4326", dst_crs, always_xy=True)
    g = shp_transform(tr.transform, geom_4326)
    polys = list(g.geoms) if g.geom_type == "MultiPolygon" else [g]
    for p in polys:
        x, y = p.exterior.xy
        ax.plot(x, y, color="red", lw=1.4, solid_capstyle="round")


def to_display(reader, rgb=False):
    """Load a reader -> (array, extent, crs) ready for imshow."""
    geo = reader.load()
    arr = np.asarray(geo.values)
    b = geo.bounds
    extent = (b[0], b[2], b[1], b[3])
    if rgb:
        im = np.moveaxis(arr[:3], 0, -1).astype("float32")
        lo, hi = np.nanpercentile(im, [2, 98])
        return np.clip((im - lo) / max(hi - lo, 1e-9), 0, 1), extent, str(geo.crs)
    a = arr.squeeze().astype("float32")
    nd = reader.nodata
    if nd is not None:
        a = np.where(a == nd, np.nan, a)
    return a, extent, str(geo.crs)


In [ ]:
import time

# The three L3A rasters worth looking at, with the outline overlaid.
# (thumbnail-grade by design — analysis-grade crops come from the L2B
# parent in the next section.)
panels = [
    ("concentrations", img.concentrations, "plasma", "full plume window"),
    ("ime_concentrations", img.ime_concentrations, "plasma",
     "IME-clipped (emission_auto integrand)"),
    ("rgb", img.rgb, None, "true-colour context"),
]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4), constrained_layout=True)
for ax, (name, reader, cmap, note) in zip(axes, panels):
    if reader is None:
        ax.set_axis_off(); ax.set_title(f"{name} (absent)"); continue
    time.sleep(1)  # be gentle with the asset gateway
    arr, extent, crs = to_display(reader, rgb=(cmap is None))
    if cmap is None:
        ax.imshow(arr, extent=extent, origin="upper")
    else:
        im = ax.imshow(arr, extent=extent, origin="upper", cmap=cmap)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="ppm·m")
    plot_outline(ax, img.outline, crs)
    h, w = (arr.shape[:2])
    ax.set_title(f"{name} — {w}×{h}px\n{note}", fontsize=9)
    ax.set_aspect("equal")
fig.suptitle(f"L3A per-plume products · {PLUME_ID}", fontsize=11)
plt.show()


## 3 · Scene tiles (L2B)

The full-swath retrieval: `cmf`, `uncertainty` (+ unortho variants),
`artifact-mask`, `uas` sidecar, and the `rgb` sibling collection.
`img.tile` resolves it from the plume's spec — probing the plume's own
version first, older candidates as backup (L3A can be re-versioned
ahead of its L2B parent; this plume is exactly that case).


In [ ]:
# One call resolves the L2B parent: the plume's own version is probed
# first, older candidates as backup (this plume is the re-versioned
# case: L3A=v3d, L2B still v3c). Lazy — nothing is read yet.
ir = img.tile
print(ir)
print(f"\nL2B collection: {str(ir.asset_paths['cmf']).split('/')[-6]}")


In [ ]:
from pathlib import Path

# Cache the three bands we plot locally, so repeated reads don't hammer
# the gateway. Idempotent — re-runs use the files.
CACHE = Path("/tmp/cm_notebooks") / PLUME_ID
CACHE.mkdir(parents=True, exist_ok=True)
for band in ("cmf", "rgb", "uncertainty"):
    local = CACHE / f"{band}.tif"
    if not local.exists():
        r = requests.get(str(ir.asset_paths[band]),
                         headers={"Authorization": f"Bearer {TOKEN}"},
                         stream=True, timeout=120)
        r.raise_for_status()
        local.write_bytes(r.content)
    ir.asset_paths[band] = str(local)
    print(f"{band:12s} {local.stat().st_size // 1024:>6d} KB")


### 3.1 Analysis-grade crops — the headline workflow

`tile_cmf` / `tile_rgb` / `tile_uncertainty` crop the L2B parent by the
plume outline at native resolution. This is the raster you quantify or
retrain on — not the L3A thumbnail.


In [ ]:
# The headline workflow: analysis-grade crops at L2B native resolution,
# next to the thumbnail-grade L3A product they replace.
crop_cmf = img.tile_cmf(pad_px=64)
crop_rgb = img.tile_rgb(pad_px=64)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)

arr, extent, crs = to_display(img.concentrations)
im = axes[0].imshow(arr, extent=extent, origin="upper", cmap="magma")
fig.colorbar(im, ax=axes[0], fraction=0.046, pad=0.04, label="ppm·m")
plot_outline(axes[0], img.outline, crs)
axes[0].set_title(f"L3A thumbnail — {arr.shape[1]}×{arr.shape[0]}px", fontsize=9)

cmf_arr = np.where(np.asarray(crop_cmf.values).squeeze() == -9999, np.nan,
                   np.asarray(crop_cmf.values).squeeze()).astype("float32")
b = crop_cmf.bounds
im = axes[1].imshow(cmf_arr, extent=(b[0], b[2], b[1], b[3]),
                    origin="upper", cmap="magma")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="ppm·m")
plot_outline(axes[1], img.outline, str(crop_cmf.crs))
axes[1].set_title(f"tile_cmf(pad_px=64) — {cmf_arr.shape[1]}×{cmf_arr.shape[0]}px "
                  f"@ L2B native", fontsize=9)

rgb_im = np.moveaxis(np.asarray(crop_rgb.values)[:3], 0, -1).astype("float32")
lo, hi = np.nanpercentile(rgb_im, [2, 98])
rgb_im = np.clip((rgb_im - lo) / max(hi - lo, 1e-9), 0, 1)
b = crop_rgb.bounds
axes[2].imshow(rgb_im, extent=(b[0], b[2], b[1], b[3]), origin="upper")
plot_outline(axes[2], img.outline, str(crop_rgb.crs))
axes[2].set_title("tile_rgb(pad_px=64) — true-colour context", fontsize=9)

for ax in axes:
    ax.set_aspect("equal")
fig.suptitle(f"Thumbnail vs analysis-grade · {PLUME_ID}", fontsize=11)
plt.show()


## 4 · Sources

A source is Carbon Mapper's DBSCAN cluster of detections at one site.
The source detail embeds every attributed plume record, so per-site
stats need one call.


In [ ]:
import pandas as pd

# Plume -> source (None if not yet DBSCAN-clustered), then per-source
# stats from the embedded plume records. Source names are re-clustered
# upstream — resolve them at run time, never hardcode.
src = api_queries.get_source_for_plume(TOKEN, PLUME_ID)
print(f"protagonist source: {src.source_name if src else '(not clustered yet)'}")

permian = api_queries.list_sources(
    TOKEN, bbox=(-104.5, 32.0, -103.5, 32.8), gas="CH4",
)
demo = max(permian, key=lambda s: s.plume_count or 0)
plumes = api_queries.list_plumes_for_source(TOKEN, demo.source_name)
em = pd.Series([p.emission_auto for p in plumes
                if p.emission_auto is not None])
pers = f"{demo.persistence:.2f}" if demo.persistence is not None else "—"
print(f"\n{demo.source_name}: {len(plumes)} detections, persistence={pers}")
print(f"emission kg/h: median={em.median():.0f}  p90={em.quantile(.9):.0f}  "
      f"max={em.max():.0f}")


## Access cheat-sheet — what works where

Verified against the live API (2026-07):

| You want | Works | Route |
|---|---|---|
| Discover recent plumes | ✅ | `/catalog/plumes/annotated` (`api_queries.list_plumes`) |
| Latest L3A per-plume products (v3c/v3d) | ✅ | asset-proxy URLs derived from the plume record (`CMPlumeImage`) |
| Latest L2B tile for a known plume/scene | ✅ | asset proxy, plume's version probed first (`img.tile`, `get_image_raster_for_plume`) |
| Anything current via **STAC** | ❌ | registry stops at `-v3a` (items end 2025-12-16) — history only |
| Browse/list recent **tiles** without a plume | ❌ | `/catalog/scenes` 401s; STAC stale — plumes are the only door to current data |
| Plumes attributed to a source | ✅ | embedded records on `/catalog/source/{name}` (`list_plumes_for_source`) |

## See also

- [`api_explore.ipynb`](api_explore.ipynb) — the typed query layer (REST + STAC, filters, exceptions).
- [Carbon Mapper reader API reference](../modules/readers_module.md#carbon-mapper-reader).
